# Sparkle V2 — Fase 3.1: TCN + attention pooling + multi-task (binaria + ordinal CORAL-aprox + auxiliares)

**Entorno de ejecución:** Google Colab (GPU T4) o Kaggle Notebooks (GPU)
**Punto de partida:** Fase 3.0 (`Sparkle_V2_Fase3_0_Binarizacion_Resplit.ipynb`) — binarización + re-split por participante.
**Objetivo de este notebook (Roadmap Fase 3.1 del plan):**

1. Reemplazar el BiGRU por una **TCN** (Conv1D dilatada + residual), que además resuelve de raíz el problema de conversión TFLite que tenían con `Bidirectional(GRU)`.
2. **Attention pooling** en vez de tomar el último estado — los pesos de atención quedan disponibles para el mapa de calor de Sparkle.
3. **Multi-task learning**: cabeza binaria (métrica principal) + cabeza ordinal (aprox. CORAL, ver nota de diseño abajo) + cabezas auxiliares (Engagement, Confusion, Frustration — se podan en el export).
4. **Verificar la conversión a TFLite int8 desde el día 1**, como pide el plan explícitamente.

**Criterio de salida de Fase 3.1:** ≥ 60–63% accuracy binaria en Test oficial; TFLite int8 < 500 kB funcionando.

---

### Nota de diseño importante — desviación respecto al CORAL "puro"

Se intentó primero la formulación CORAL estándar (Cao et al. 2020): un único logit con **peso compartido** entre los `K-1` umbrales + un vector de sesgos con orden garantizado (softplus + cumsum), broadcast a `K-1` salidas. **Esta formulación rompe la conversión a TFLite** (`'tfl.fully_connected' op num_input_elements / z_in != num_output_elements / z_out`), tanto con broadcast implícito como con `tf.tile` explícito — se probaron ambas variantes y ambas fallan. La causa es la interacción entre el batch_size fijo (necesario para evitar el mismo problema de `TensorListReserve` que ya sufrieron con `Bidirectional(GRU)`) y la fusión de la capa `Dense(1)` con la operación de broadcast posterior.

**Solución adoptada:** `Dense(K-1)` con pesos **independientes** por umbral (sin compartir peso). Convierte limpio a TFLite (verificado en este notebook). Se pierde la garantía arquitectónica de monotonicidad estricta de CORAL puro, pero el loss ordinal (suma de BCE sobre `y > k` para cada umbral) sigue empujando en la práctica a que `P(y>0) ≥ P(y>1)`. **Mitigación para producción:** aplicar `cummin` (mínimo acumulado) sobre las `K-1` probabilidades en el código de inferencia (fuera del grafo TFLite, es una operación de 2 elementos) para garantizar monotonicidad estricta sin costo de conversión ni de latencia.

## Cómo correr esto en Colab

1. `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → **T4 GPU**.
2. Ejecutar la celda de setup: monta Drive y verifica que estén los datos.
3. `Entorno de ejecución` → `Ejecutar todas`.

**No hace falta `pip install`**: Colab ya trae TensorFlow, Keras 3, scikit-learn y pandas. Instalar/reinstalar TensorFlow rompe la sesión y obliga a reiniciar el runtime.

> **Adaptación a Keras 3.** Colab hoy corre TensorFlow 2.19 + Keras 3. Respecto de la versión original de este notebook (escrita contra Keras 2) cambian tres cosas, señalizadas en el código: (a) el export a TFLite pasa por `ExportArchive` → SavedModel → `from_saved_model` (en Keras 3 `TFLiteConverter.from_keras_model` ya no acepta modelos Keras directamente), (b) las capas custom declaran sus sub-capas en `__init__` y exponen `compute_output_shape`/`get_config`, (c) todas las capas llevan nombre explícito para que la copia de pesos al modelo de export sea verificable.

In [ ]:
# =====================================================================
# SETUP — detección de entorno, montaje de Drive y verificación de datos
# =====================================================================
import sys, os, glob, json, time

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
elif os.path.isdir('/kaggle/input'):
    PROJECT_DIR = '/kaggle/input/sparklev2'
else:
    PROJECT_DIR = os.environ.get('SPARKLE_PROJECT_DIR', './SparkleV2')

print('Entorno:', 'Colab' if IN_COLAB else ('Kaggle' if os.path.isdir('/kaggle/input') else 'local'))
print('PROJECT_DIR =', PROJECT_DIR)
assert os.path.isdir(PROJECT_DIR), (
    f'No existe {PROJECT_DIR}. En Colab la carpeta tiene que estar en "Mi unidad" '
    f'(los accesos directos a "Compartido conmigo" NO se montan). Ajustá PROJECT_DIR a mano si hace falta.')

FEATURES_DIR = f'{PROJECT_DIR}/features_v2'
OUT_DIR      = f'{PROJECT_DIR}/dataset_final_fase3'
CKPT_DIR     = f'{PROJECT_DIR}/checkpoints_fase3_1'
os.makedirs(OUT_DIR,  exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

for split in ('Train', 'Validation', 'Test'):
    n = len(glob.glob(f'{FEATURES_DIR}/{split}/*.npz'))
    print(f'features_v2/{split}: {n} clips')
    assert n > 0, f'No hay .npz en {FEATURES_DIR}/{split}'

import tempfile, shutil
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from collections import Counter
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             classification_report, confusion_matrix)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

KERAS3 = keras.__version__.startswith('3')
gpus = tf.config.list_physical_devices('GPU')
print(f'TF {tf.__version__} | Keras {keras.__version__} | GPU: {gpus}')
if not gpus:
    print('\n*** SIN GPU: Entorno de ejecucion -> Cambiar tipo de entorno -> T4 GPU ***\n')

SEED = 42
N_FRAMES = 60
np.random.seed(SEED); tf.random.set_seed(SEED)

MAX_EPOCHS = 120
PATIENCE   = 15
BATCH_SIZE = 128
SEEDS      = [0, 1, 2, 3, 4]

## 1. Datos — re-split por participante + labels auxiliares

Mismo tratamiento de Fase 3.0 (Paso B: NaN → deltas curados → 32 features → `StandardScaler` fit en train del fold/pool), pero ahora se cargan también `engagement`, `confusion` y `frustration` por clip (ya están en cada `.npz` de `features_v2`, según documenta el notebook de Fase 2) para las cabezas auxiliares de multi-task.

In [ ]:
DELTA_FEATURES = ['ear_avg', 'yaw', 'pitch', 'mar', 'jawOpen',
                  'eyeBlinkLeft', 'eyeBlinkRight',
                  'eyeLookInLeft', 'eyeLookOutLeft', 'browInnerUp']

def load_raw_split_full(split):
    """Devuelve X (N,60,22), boredom 3-niveles, engagement/confusion/frustration
    (0-3 crudos, para las cabezas auxiliares) e ids."""
    files = sorted(glob.glob(f'{FEATURES_DIR}/{split}/*.npz'))
    X, y3, eng, conf, frus, ids = [], [], [], [], [], []
    feat_names = None
    for f in files:
        dd = np.load(f, allow_pickle=True)
        seq = dd['sequence'].astype(np.float32)
        if seq.shape != (N_FRAMES, 22):
            continue
        if feat_names is None:
            feat_names = list(dd['feature_names'])
        X.append(seq)
        b = int(dd['boredom'])
        y3.append(0 if b == 0 else (1 if b == 1 else 2))
        eng.append(int(dd['engagement']))
        conf.append(int(dd['confusion']))
        frus.append(int(dd['frustration']))
        ids.append(os.path.splitext(os.path.basename(f))[0])
    return (np.stack(X), np.array(y3, dtype=np.int64), np.array(eng), np.array(conf),
            np.array(frus), np.array(ids), feat_names)

X_tr_raw, y3_tr, eng_tr, conf_tr, frus_tr, id_tr, FEATS22 = load_raw_split_full('Train')
X_va_raw, y3_va, eng_va, conf_va, frus_va, id_va, _        = load_raw_split_full('Validation')

X_pool_raw = np.concatenate([X_tr_raw, X_va_raw], axis=0)
y3_pool    = np.concatenate([y3_tr, y3_va])
eng_pool   = np.concatenate([eng_tr, eng_va])
conf_pool  = np.concatenate([conf_tr, conf_va])
frus_pool  = np.concatenate([frus_tr, frus_va])
id_pool    = np.concatenate([id_tr, id_va])
participant_pool = np.array([cid[:6] for cid in id_pool])
y_bin_pool = (y3_pool > 0).astype(np.int64)

print(f'Pool Train+Validation: {len(id_pool)} clips, '
      f'{len(set(participant_pool.tolist()))} participantes')
print('Dist binaria:', Counter(y_bin_pool.tolist()), '| Dist 3 niveles:', Counter(y3_pool.tolist()))

In [ ]:
def interpolate_sequence(seq):
    df = pd.DataFrame(seq)
    df = df.interpolate(method='linear', axis=0, limit_direction='both')
    return df.ffill().bfill().values.astype(np.float32)

def clean_raw(X, train_medians=None, nan_frame_thresh=0.30):
    keep, X_clean = [], []
    for i in range(len(X)):
        if np.isnan(X[i]).any(axis=1).mean() > nan_frame_thresh:
            continue
        seq = interpolate_sequence(X[i])
        if np.isnan(seq).any() and train_medians is not None:
            idx = np.where(np.isnan(seq))
            seq[idx] = np.take(train_medians, idx[1])
        X_clean.append(seq); keep.append(i)
    return np.stack(X_clean), np.array(keep)

delta_idx = [FEATS22.index(f) for f in DELTA_FEATURES]
def add_deltas(X, idx):
    d_ = np.zeros_like(X[:, :, idx])
    d_[:, 1:, :] = np.diff(X[:, :, idx], axis=1)
    return np.concatenate([X, d_], axis=-1)

def sliding_windows_multi(X, labels_dict, win_lens=(40, 44, 48), stride=8, n_frames=60):
    """Igual que en Fase 3.0 pero propaga TODAS las etiquetas (binaria, 3 niveles,
    engagement, confusion, frustration) a cada sub-ventana generada."""
    Xw = []
    yw = {k: [] for k in labels_dict}
    for i in range(len(X)):
        for L in win_lens:
            for start in range(0, n_frames - L + 1, stride):
                sub = X[i, start:start+L, :]
                pad = np.zeros((n_frames - L, X.shape[-1]), dtype=X.dtype)
                Xw.append(np.concatenate([sub, pad], axis=0))
                for k, arr in labels_dict.items():
                    yw[k].append(arr[i])
    return np.stack(Xw), {k: np.array(v) for k, v in yw.items()}

train_medians_full = np.nanmedian(X_pool_raw.reshape(-1, 22), axis=0)
X_pool_c, keep_pool = clean_raw(X_pool_raw, train_medians_full)
pool_labels = {'y_bin': y_bin_pool[keep_pool], 'y3': y3_pool[keep_pool],
               'eng': eng_pool[keep_pool], 'conf': conf_pool[keep_pool], 'frus': frus_pool[keep_pool]}
X_pool_d = add_deltas(X_pool_c, delta_idx)

scaler_final = StandardScaler().fit(X_pool_d.reshape(-1, X_pool_d.shape[-1]))
def scale_final(X):
    N, T, F = X.shape
    return scaler_final.transform(X.reshape(-1, F)).reshape(N, T, F).astype(np.float32)
X_pool_s = scale_final(X_pool_d)

X_pool_aug, y_pool_aug = sliding_windows_multi(X_pool_s, pool_labels)
print(f'Pool tras augmentación: {X_pool_s.shape[0]} -> {X_pool_aug.shape[0]} secuencias')

# Split train/val (respetando participante) SOLO para early stopping del modelo final
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tr_idx0, va_idx0 = next(sgkf.split(X_pool_c, pool_labels['y_bin'],
                                    groups=participant_pool[keep_pool]))
# Nota: el split se calcula sobre el pool SIN augmentar (por clip) y luego se
# proyecta a los indices aumentados para no filtrar el mismo clip origen entre train/val.
clip_to_aug = []
n_win_per_clip = X_pool_aug.shape[0] // X_pool_s.shape[0]
for clip_i in range(X_pool_s.shape[0]):
    clip_to_aug.extend([clip_i] * n_win_per_clip)
clip_to_aug = np.array(clip_to_aug)
aug_is_train = np.isin(clip_to_aug, tr_idx0)

X_tr_final = X_pool_aug[aug_is_train]
X_va_final = X_pool_s[va_idx0]  # validation SIN augmentar (evaluación limpia)
y_tr_final = {k: v[aug_is_train] for k, v in y_pool_aug.items()}
y_va_final = {k: v[va_idx0] for k, v in pool_labels.items()}
print(f'Train final (aug): {len(X_tr_final)} | Val final (sin aug): {len(X_va_final)}')

In [ ]:
# Test oficial: mismo tratamiento (medianas y scaler del pool completo)
X_te_raw, y3_te, eng_te, conf_te, frus_te, id_te, _ = load_raw_split_full('Test')
y_bin_te = (y3_te > 0).astype(np.int64)
X_te_c, keep_te = clean_raw(X_te_raw, train_medians_full)
X_te_d = add_deltas(X_te_c, delta_idx)
X_te_s = scale_final(X_te_d)
y_te_final = {'y_bin': y_bin_te[keep_te], 'y3': y3_te[keep_te],
              'eng': eng_te[keep_te], 'conf': conf_te[keep_te], 'frus': frus_te[keep_te]}

trivial_test = max(Counter(y_te_final['y_bin'].tolist()).values()) / len(y_te_final['y_bin'])
print(f"Test oficial: n={len(y_te_final['y_bin'])} | trivial={trivial_test:.4f}")

## 2. Arquitectura — TCN residual + attention pooling + multi-task

Bloques residuales con `Conv1D` causal dilatada (dilataciones 1, 2, 4, 8 → campo receptivo cubre toda la secuencia de 60 pasos), 64 filtros, kernel 3. `AttentionPooling` reemplaza al último estado recurrente. Cinco cabezas: binaria (principal), ordinal aproximado (2 salidas, ver nota de diseño arriba), y tres auxiliares de multi-task (podadas en el export).

**Adaptación a Keras 3.** Dos cambios respecto de la versión original:

- `AttentionPooling` y `CoralHead` crean sus sub-capas en `__init__` (no en `build`), declaran `compute_output_shape` y `get_config`, y se registran como serializables — así el modelo se puede guardar y volver a cargar desde `.keras` sin `custom_objects` a mano.
- **Todas** las capas del tronco TCN llevan nombre explícito (`tcn0_conv1`, `tcn0_bn1`, …). La versión original copiaba los pesos al modelo de export *por nombre*, pero los nombres autogenerados (`conv1d_7`, `conv1d_23`, …) dependen de un contador global y **no coinciden** entre el modelo entrenado y el de export: el tronco TCN se habría exportado con pesos aleatorios sin que nada fallara. Con nombres explícitos la copia es exacta, y la celda de export lo verifica con un `assert`.

In [ ]:
@keras.saving.register_keras_serializable(package='sparkle')
class AttentionPooling(layers.Layer):
    '''Softmax aprendido sobre los 60 pasos temporales; los pesos son interpretables
    para el mapa de calor de Sparkle (se expone como segunda salida).'''
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.score_dense = layers.Dense(1, name='attn_score')
    def call(self, x):
        scores = self.score_dense(x)
        weights = tf.nn.softmax(scores, axis=1)
        pooled = tf.reduce_sum(x * weights, axis=1)
        return pooled, tf.squeeze(weights, axis=-1)
    # Sin compute_output_shape a proposito: esta capa devuelve DOS tensores y Keras 3
    # interpreta una tupla devuelta por compute_output_shape como UNA sola forma. Al
    # omitirlo, Keras infiere ambas salidas trazando call() simbolicamente, que es lo
    # correcto para salidas multiples.

@keras.saving.register_keras_serializable(package='sparkle')
class CoralHead(layers.Layer):
    '''Aprox. CORAL: Dense(K-1) con pesos independientes por umbral (ver nota de
    diseno arriba sobre por que NO se comparte el peso: rompe TFLite).'''
    def __init__(self, num_classes, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.dense = layers.Dense(num_classes - 1, name='coral_thresholds')
    def call(self, x):
        return tf.nn.sigmoid(self.dense(x))
    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.num_classes - 1)
    def get_config(self):
        cfg = super().get_config(); cfg.update({'num_classes': self.num_classes}); return cfg

def tcn_residual_block(x, filters, kernel_size, dilation_rate, dropout=0.2, l2=1e-4, prefix='tcn'):
    '''Bloque residual con Conv1D causal dilatada. Nombres explicitos: la copia de
    pesos al modelo de export depende de que coincidan uno a uno.'''
    reg = keras.regularizers.l2(l2) if l2 else None
    prev = x
    h = layers.Conv1D(filters, kernel_size, padding='causal', dilation_rate=dilation_rate,
                      kernel_regularizer=reg, name=f'{prefix}_conv1')(x)
    h = layers.BatchNormalization(name=f'{prefix}_bn1')(h)
    h = layers.Activation('relu', name=f'{prefix}_act1')(h)
    h = layers.Dropout(dropout, name=f'{prefix}_do1')(h)
    h = layers.Conv1D(filters, kernel_size, padding='causal', dilation_rate=dilation_rate,
                      kernel_regularizer=reg, name=f'{prefix}_conv2')(h)
    h = layers.BatchNormalization(name=f'{prefix}_bn2')(h)
    h = layers.Activation('relu', name=f'{prefix}_act2')(h)
    h = layers.Dropout(dropout, name=f'{prefix}_do2')(h)
    if prev.shape[-1] != filters:
        prev = layers.Conv1D(filters, 1, padding='same', name=f'{prefix}_proj')(prev)
    add = layers.Add(name=f'{prefix}_add')([prev, h])
    return layers.Activation('relu', name=f'{prefix}_out')(add)

def ordinal_loss(y_true_3level, p_hat, n_classes=3):
    '''y_true_3level: (batch,) enteros 0..K-1. Se extiende a K-1 targets binarios
    y_k = 1[y > k] y se promedia BCE sobre los K-1 umbrales.'''
    y_true_3level = tf.cast(y_true_3level, tf.float32)
    losses = []
    for k in range(n_classes - 1):
        yk = tf.cast(y_true_3level > k, tf.float32)
        losses.append(keras.losses.binary_crossentropy(yk, p_hat[:, k]))
    return tf.reduce_mean(tf.add_n(losses) / (n_classes - 1))

def build_tcn_multitask(input_shape, n_boredom_classes=3, filters=64, kernel_size=3,
                        dilations=(1, 2, 4, 8), dropout=0.3, l2=1e-4, batch_size=None,
                        export=False):
    '''export=True -> solo las 2 cabezas de produccion (binaria + ordinal) + los pesos
    de atencion, sin regularizacion ni dropout, con batch fijo para TFLite.'''
    if export:
        dropout, l2 = 0.0, 0.0
    inp = keras.Input(shape=input_shape, batch_size=batch_size, name='features')
    x = inp
    for i, d in enumerate(dilations):
        x = tcn_residual_block(x, filters, kernel_size, d, dropout=dropout, l2=l2, prefix=f'tcn{i}')
    pooled, attn_weights = AttentionPooling(name='attention_pooling')(x)
    h = layers.Dense(32, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(l2) if l2 else None,
                     name='head_dense')(pooled)
    if not export:
        h = layers.Dropout(dropout, name='head_dropout')(h)
    binary_out = layers.Dense(1, activation='sigmoid', name='binary')(h)
    coral_out  = CoralHead(n_boredom_classes, name='coral')(h)
    if export:
        return keras.Model(inp, [binary_out, coral_out, attn_weights], name='tcn_export')
    aux_engagement  = layers.Dense(4, activation='softmax', name='aux_engagement')(h)
    aux_confusion   = layers.Dense(4, activation='softmax', name='aux_confusion')(h)
    aux_frustration = layers.Dense(4, activation='softmax', name='aux_frustration')(h)
    return keras.Model(inp, [binary_out, coral_out, aux_engagement, aux_confusion,
                             aux_frustration], name='tcn_multitask')

_ref = build_tcn_multitask((N_FRAMES, X_pool_aug.shape[-1]))
print(f'{_ref.count_params():,} parametros totales (objetivo del plan: ~50-80k en el tronco TCN; '
      f'el total incluye 5 cabezas, se poda a 2 en el export)')
_ref_export = build_tcn_multitask((N_FRAMES, X_pool_aug.shape[-1]), export=True, batch_size=1)
print(f'{_ref_export.count_params():,} parametros en el modelo de export (2 cabezas)')
del _ref, _ref_export

## 3. Entrenamiento — 5 semillas, pérdida combinada multi-task

`loss_total = loss_binaria + 0.5·loss_ordinal + 0.1·(loss_engagement + loss_confusion + loss_frustration)`. Los pesos auxiliares bajos (0,1) siguen la recomendación del plan (§3.3): regularizan sin dominar el objetivo principal. Se usa un `train_step` manual (en vez de `model.compile`/`fit` estándar) porque las pérdidas combinan BCE, la pérdida ordinal custom y categorical crossentropy sobre 5 salidas distintas.

**Adaptación a Keras 3:** el `train_step` decorado con `@tf.function` se define **dentro** de `train_one_seed`. En la versión original era global y recibía `model`/`optimizer` como argumentos: al entrar la segunda semilla, el optimizador nuevo intenta crear sus variables de estado en un retrace posterior al primero y TF lo rechaza (`tf.function only supports singleton tf.Variables created on the first call`). Con el cierre por semilla cada `tf.function` es nueva y el problema desaparece.

In [ ]:
def smoothed_class_weights(y, beta=0.5):
    counts = Counter(y.tolist()); total = len(y)
    w = {int(c): (total / counts[c]) ** beta for c in counts}
    mean_w = np.mean(list(w.values()))
    return {c: w[c] / mean_w for c in w}

def make_dataset(X, y_dict, class_weight_bin, batch_size=BATCH_SIZE, shuffle=True, seed=SEED):
    sw = np.array([class_weight_bin[int(v)] for v in y_dict['y_bin']], dtype='float32')
    ds = tf.data.Dataset.from_tensor_slices((
        X.astype('float32'),
        y_dict['y_bin'].astype('float32'),
        y_dict['y3'].astype('int64'),
        keras.utils.to_categorical(y_dict['eng'], 4).astype('float32'),
        keras.utils.to_categorical(y_dict['conf'], 4).astype('float32'),
        keras.utils.to_categorical(y_dict['frus'], 4).astype('float32'),
        sw,
    ))
    if shuffle:
        ds = ds.shuffle(4096, seed=seed, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

def evaluate_binary(model, X, y_bin, batch_size=256):
    p_bin = model.predict(X, verbose=0, batch_size=batch_size)[0].ravel()
    p_bin_cls = (p_bin > 0.5).astype(int)
    return p_bin_cls, {
        'acc': accuracy_score(y_bin, p_bin_cls),
        'f1_macro': f1_score(y_bin, p_bin_cls, average='macro'),
        'balanced_acc': balanced_accuracy_score(y_bin, p_bin_cls),
    }

# ---------------------------------------------------------------------------
# Protocolo de entrenamiento PARAMETRIZADO.
# La primera corrida de 3.1 (lr=1e-3, aux=0.1, sin scheduler, sin min_epochs)
# dio 0.5824 +/- 0.0277, por debajo del BiGRU de 3.0 (0.6088 +/- 0.0066), y en
# 2 de 5 semillas el mejor val_f1 cayo en la EPOCA 0 -> restore_best_weights
# devolvia pesos sin entrenar. Los cuatro parametros de abajo son los que
# corrigen eso; el barrido de la celda siguiente elige la configuracion.
# ---------------------------------------------------------------------------
CFG_BASE = dict(
    lr=5e-4,             # era 1e-3 (el BiGRU de 3.0 usaba 5e-4)
    aux_weight=0.1,      # peso de las 3 cabezas auxiliares
    ord_weight=0.5,      # peso de la perdida ordinal
    min_epochs=10,       # no habilitar early stopping antes de esta epoca
    lr_patience=6,       # ReduceLROnPlateau manual (el fit() de 3.0 lo tenia)
    lr_factor=0.5,
    lr_min=1e-5,
    label_smoothing=0.1, # la version original no lo aplicaba a la cabeza binaria
    dropout=0.3,
    filters=64,
)

def train_one_seed(seed, cfg=None, epochs=MAX_EPOCHS, patience=PATIENCE,
                   batch_size=BATCH_SIZE, verbose=True):
    cfg = {**CFG_BASE, **(cfg or {})}
    try:
        keras.utils.set_random_seed(seed)
    except AttributeError:
        pass
    tf.random.set_seed(seed); np.random.seed(seed)

    model = build_tcn_multitask((N_FRAMES, X_tr_final.shape[-1]),
                                filters=cfg['filters'], dropout=cfg['dropout'])
    optimizer = keras.optimizers.Adam(learning_rate=cfg['lr'], clipnorm=1.0)
    cw = smoothed_class_weights(y_tr_final['y_bin'], beta=0.5)
    ds_train = make_dataset(X_tr_final, y_tr_final, cw, batch_size=batch_size, seed=seed)

    aux_w, ord_w, ls = cfg['aux_weight'], cfg['ord_weight'], cfg['label_smoothing']

    # tf.function LOCAL a esta semilla (ver nota de arriba sobre variables del optimizador)
    @tf.function
    def train_step(Xb, yb_bin, yb_3, yb_eng, yb_conf, yb_frus, sw):
        with tf.GradientTape() as tape:
            p_bin, p_coral, p_eng, p_conf, p_frus = model(Xb, training=True)
            l_bin = tf.reduce_mean(sw * keras.losses.binary_crossentropy(
                yb_bin[:, None], p_bin, label_smoothing=ls))
            l_ord = ordinal_loss(yb_3, p_coral)
            total = l_bin + ord_w * l_ord
            if aux_w > 0:
                l_eng  = tf.reduce_mean(keras.losses.categorical_crossentropy(yb_eng, p_eng))
                l_conf = tf.reduce_mean(keras.losses.categorical_crossentropy(yb_conf, p_conf))
                l_frus = tf.reduce_mean(keras.losses.categorical_crossentropy(yb_frus, p_frus))
                total += aux_w * (l_eng + l_conf + l_frus)
            if model.losses:
                total += tf.add_n(model.losses)        # regularizacion L2
        grads = tape.gradient(total, model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return total

    def set_lr(v):
        try:
            optimizer.learning_rate.assign(v)
        except AttributeError:
            optimizer.learning_rate = v

    best_f1, wait, lr_wait, best_weights, best_epoch = -1.0, 0, 0, None, -1
    hist = []
    t0 = time.time()
    for epoch in range(epochs):
        for batch in ds_train:
            train_step(*batch)
        _, vm = evaluate_binary(model, X_va_final, y_va_final['y_bin'])
        f1 = vm['f1_macro']; hist.append(f1)

        # Antes de min_epochs no se registra "mejor" ni corre la paciencia: evita
        # que restore_best_weights devuelva un modelo practicamente sin entrenar.
        if epoch < cfg['min_epochs']:
            if verbose and epoch % 5 == 0:
                print(f'    epoca {epoch:3d} | val_f1={f1:.4f} | (calentamiento, '
                      f'min_epochs={cfg["min_epochs"]}) | {(time.time()-t0)/60:.1f} min')
            continue

        if best_weights is None or f1 > best_f1 + 1e-4:
            best_f1, wait, lr_wait = f1, 0, 0
            best_weights, best_epoch = model.get_weights(), epoch
        else:
            wait += 1; lr_wait += 1
            if lr_wait >= cfg['lr_patience']:        # ReduceLROnPlateau manual
                nueva = max(float(optimizer.learning_rate) * cfg['lr_factor'], cfg['lr_min'])
                set_lr(nueva); lr_wait = 0
                if verbose:
                    print(f'    epoca {epoch:3d} | lr -> {nueva:.2e}')
        if verbose and (epoch % 10 == 0 or wait >= patience):
            print(f'    epoca {epoch:3d} | val_f1={f1:.4f} | mejor={best_f1:.4f} '
                  f'(ep {best_epoch}) | {(time.time()-t0)/60:.1f} min')
        if wait >= patience:
            break

    model.set_weights(best_weights)
    return model, best_f1, epoch + 1, (time.time() - t0) / 60, best_epoch, hist

def append_csv(row, path):
    pd.DataFrame([row]).to_csv(path, mode='a', header=not os.path.exists(path), index=False)

def correr_config(nombre, cfg, seeds, csv_path=None, verbose=False):
    '''Entrena `seeds` semillas con una configuracion y devuelve (df, modelo_seed0).'''
    filas, modelo0 = [], None
    for s in seeds:
        model, best_val_f1, ep_corridas, minutos, best_ep, _ = train_one_seed(
            s, cfg=cfg, verbose=verbose)
        _, m = evaluate_binary(model, X_te_s, y_te_final['y_bin'])
        m.update({'config': nombre, 'seed': s, 'epocas': ep_corridas,
                  'mejor_epoca': best_ep, 'best_val_f1': round(best_val_f1, 4),
                  'gap_vs_trivial': m['acc'] - trivial_test, 'minutos': round(minutos, 2)})
        filas.append(m)
        if csv_path:
            append_csv(m, csv_path)
        print(f"  [{nombre}] seed={s}: test_acc={m['acc']:.4f} f1={m['f1_macro']:.4f} "
              f"gap={m['gap_vs_trivial']:+.4f} | mejor_epoca={best_ep} de {ep_corridas} "
              f"({minutos:.1f} min)")
        if modelo0 is None:
            modelo0 = model
    return pd.DataFrame(filas), modelo0

### 3.1 Diagnóstico — barrido corto para elegir el protocolo

La primera corrida de este notebook quedó en **0,5824 ± 0,0277**, por debajo del BiGRU de Fase 3.0 (0,6088 ± 0,0066) y sin cumplir el criterio de salida (≥ 60-63 %). El síntoma decisivo: en las semillas 3 y 4 el mejor `val_f1` ocurrió en la **época 0**, o sea que `restore_best_weights` devolvió pesos sin entrenar — y aun así fueron dos de los tres mejores resultados. Eso no es un modelo que aprende.

Diagnóstico: la TCN tiene 99k parámetros contra ~14k del BiGRU (7×) sobre una señal débil; las 51.096 secuencias son 6.388 clips reales de 82 participantes (la augmentación por ventanas no agrega información). Sobreajusta en 3-4 épocas, y el loop manual no tenía scheduler de learning rate ni piso de épocas.

Este barrido prueba tres configuraciones con **2 semillas cada una** (~15-20 min en total) para decidir con datos:

| Config | Qué cambia respecto de la corrida original | Hipótesis que testea |
| :--- | :--- | :--- |
| `B_protocolo` | lr 5e-4 · scheduler · min_epochs 10 · label smoothing | ¿era solo el protocolo de entrenamiento? |
| `C_sin_aux` | B + peso de cabezas auxiliares a 0 | ¿las tareas auxiliares meten ruido en el tronco? |
| `D_chico` | C + 32 filtros (≈ 30k parámetros) | ¿el problema es exceso de capacidad? |

**Criterio de decisión:** la configuración ganadora tiene que alcanzar **paridad con el BiGRU (~0,60-0,61)** y, sobre todo, mostrar `mejor_epoca` bien lejos del piso de `min_epochs`. Si ninguna llega, se avanza a Fase 3.2 con el BiGRU como tronco y se documenta la TCN como alternativa descartada.

In [ ]:
CONFIGS_DIAG = {
    'B_protocolo': dict(),                                    # CFG_BASE tal cual
    'C_sin_aux':   dict(aux_weight=0.0),
    'D_chico':     dict(aux_weight=0.0, filters=32, dropout=0.35),
}
SEEDS_DIAG = [0, 1]        # 2 semillas por config para que sea rapido

csv_diag = f'{CKPT_DIR}/fase3_1_diagnostico.csv'
if os.path.exists(csv_diag):
    os.remove(csv_diag)

diag = []
for nombre, cfg in CONFIGS_DIAG.items():
    n_par = build_tcn_multitask((N_FRAMES, X_tr_final.shape[-1]),
                                filters={**CFG_BASE, **cfg}['filters'],
                                dropout={**CFG_BASE, **cfg}['dropout']).count_params()
    print(f'\n--- {nombre} ({n_par:,} parametros) --- {cfg or "CFG_BASE"}')
    df_cfg, _ = correr_config(nombre, cfg, SEEDS_DIAG, csv_diag)
    diag.append(df_cfg)

df_diag = pd.concat(diag, ignore_index=True)
resumen = df_diag.groupby('config').agg(
    acc_media=('acc', 'mean'), acc_std=('acc', 'std'),
    f1_medio=('f1_macro', 'mean'), gap=('gap_vs_trivial', 'mean'),
    mejor_epoca=('mejor_epoca', 'mean'), min=('minutos', 'mean')).round(4)

print('\n' + '=' * 78)
print('DIAGNOSTICO FASE 3.1 -- referencias: BiGRU 3.0 = 0.6088 | TCN original = 0.5824')
print('=' * 78)
print(resumen)

mejor = resumen['acc_media'].idxmax()
print(f'\nMejor configuracion: {mejor} (acc={resumen.loc[mejor, "acc_media"]:.4f})')
if resumen.loc[mejor, 'acc_media'] >= 0.60:
    print('-> Alcanza paridad con el BiGRU. Usar esta config en la corrida final Y en 3.2b.')
else:
    print('-> NO alcanza paridad con el BiGRU (0.6088). Opciones: (a) barrer mas configs,')
    print('   (b) avanzar a Fase 3.2 con el BiGRU como tronco y documentar la TCN como')
    print('   alternativa descartada. No conviene seguir invirtiendo aca: la palanca es 3.2.')
if resumen.loc[mejor, 'mejor_epoca'] <= CFG_BASE['min_epochs'] + 1:
    print('\n*** ATENCION: la mejor epoca sigue pegada al piso de min_epochs. El modelo no')
    print('    esta aprendiendo, solo se lo esta frenando antes. Revisar lr y capacidad. ***')

### 3.2 Corrida final con la configuración elegida

Poné en `CFG_FINAL` la configuración ganadora del barrido (o dejá `None` para usar `CFG_BASE`) y corré las 5 semillas. **Anotá cuál elegiste**: la misma tiene que ir al notebook 3.2b, porque el modelo de fusión usa este mismo tronco y este mismo loop de entrenamiento.

In [ ]:
CFG_FINAL = CONFIGS_DIAG[mejor]      # o fijala a mano: dict(aux_weight=0.0), etc.
print('Configuracion final:', {**CFG_BASE, **CFG_FINAL})

csv_31 = f'{CKPT_DIR}/fase3_1_resultados_5semillas.csv'
if os.path.exists(csv_31):
    os.remove(csv_31)

df_31, best_model_31 = correr_config('final', CFG_FINAL, SEEDS, csv_31, verbose=True)
best_model_31.save(f'{CKPT_DIR}/tcn_multitask_seed0.keras')

print('\n=== RESUMEN FASE 3.1 (TCN multi-task, 5 semillas, Test oficial) ===')
print(f"Accuracy:  {df_31['acc'].mean():.4f} +/- {df_31['acc'].std():.4f}  (trivial={trivial_test:.4f})")
print(f"F1 macro:  {df_31['f1_macro'].mean():.4f} +/- {df_31['f1_macro'].std():.4f}")
print(f"Bal. acc:  {df_31['balanced_acc'].mean():.4f} +/- {df_31['balanced_acc'].std():.4f}")
print(f"Gap vs trivial: {df_31['gap_vs_trivial'].mean():+.4f} +/- {df_31['gap_vs_trivial'].std():.4f}")
print(f"Mejor epoca (media): {df_31['mejor_epoca'].mean():.1f} | "
      f"epocas corridas (media): {df_31['epocas'].mean():.1f}")

p_te_best, _ = evaluate_binary(best_model_31, X_te_s, y_te_final['y_bin'])
print(classification_report(y_te_final['y_bin'], p_te_best,
                            target_names=['Atento','Desatencion'], digits=3))

print('\n--- Comparacion con Fase 3.0 ---')
print(f"BiGRU (3.0 Paso B):  0.6088 +/- 0.0066  gap +0.0648")
print(f"TCN (3.1 original):  0.5824 +/- 0.0277  gap +0.0385")
print(f"TCN (3.1 corregida): {df_31['acc'].mean():.4f} +/- {df_31['acc'].std():.4f}  "
      f"gap {df_31['gap_vs_trivial'].mean():+.4f}")
print('\nRECORDAR: copiar CFG_FINAL al notebook 3.2b antes de correr la fusion.')

## 4. Export a TFLite — verificación desde el día 1 (binaria + ordinal, sin cabezas auxiliares)

Se reconstruye un modelo de **batch fijo** (igual mitigación que ya usan para el BiGRU) con solo las 2 cabezas de producción (binaria + ordinal aproximado); las cabezas auxiliares de multi-task se podan porque solo aportan regularización en entrenamiento.

**Adaptación a Keras 3 (crítica).** En TF ≥ 2.16 `tf.lite.TFLiteConverter.from_keras_model()` ya no acepta un modelo Keras 3. El camino soportado es exportar primero a SavedModel y convertir desde ahí. Se usa `keras.export.ExportArchive` en vez de `model.export()` porque permite fijar explícitamente la firma de entrada con **batch = 1**, que es justo la mitigación que ya venían usando para evitar operadores dinámicos (`TensorListReserve`) en la conversión.

In [ ]:
def keras_to_tflite(model, input_specs, representative_dataset=None, verbose=True):
    '''Convierte un modelo Keras 3 a TFLite. Camino: ExportArchive -> SavedModel con
    firma de batch fijo -> TFLiteConverter.from_saved_model. Cae a from_keras_model
    si el entorno todavia corriera Keras 2.'''
    sm_dir = tempfile.mkdtemp(prefix='sparkle_sm_')
    shutil.rmtree(sm_dir, ignore_errors=True)
    exported = False
    try:
        from keras.export import ExportArchive
        archive = ExportArchive()
        archive.track(model)
        if len(input_specs) == 1:
            fn = lambda x: model(x, training=False)
        else:
            fn = lambda *xs: model(list(xs), training=False)
        archive.add_endpoint(name='serve', fn=fn, input_signature=list(input_specs))
        archive.write_out(sm_dir)
        exported = True
        if verbose:
            print('  SavedModel exportado con ExportArchive (batch fijo)')
    except Exception as e:
        print('  ExportArchive no disponible o fallo:', repr(e)[:200])
        try:
            model.export(sm_dir); exported = True
            print('  SavedModel exportado con model.export()')
        except Exception as e2:
            print('  model.export() fallo:', repr(e2)[:200])

    if exported:
        converter = tf.lite.TFLiteConverter.from_saved_model(sm_dir)
    else:
        print('  Fallback: from_keras_model (solo funciona con Keras 2)')
        converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    if representative_dataset is not None:
        converter.representative_dataset = representative_dataset
    tflite_bytes = converter.convert()
    shutil.rmtree(sm_dir, ignore_errors=True)
    return tflite_bytes

def copy_weights_by_name(src, dst, allow_missing=()):
    '''Copia pesos capa a capa por nombre y VERIFICA que no haya quedado ninguna
    capa con pesos sin copiar (con nombres autogenerados esto fallaba en silencio
    y el tronco TCN se exportaba con pesos aleatorios).'''
    src_map = {l.name: l for l in src.layers}
    copied, missing = [], []
    for layer in dst.layers:
        if not layer.weights:
            continue
        s = src_map.get(layer.name)
        if s is None or not s.weights:
            missing.append(layer.name); continue
        try:
            layer.set_weights(s.get_weights()); copied.append(layer.name)
        except ValueError:
            missing.append(f'{layer.name}(shape)')
    print(f'  Pesos copiados: {len(copied)} capas | sin copiar: {missing or "ninguna"}')
    pendientes = [m for m in missing if m not in allow_missing]
    assert not pendientes, f'Capas sin pesos copiados en el modelo de export: {pendientes}'
    return copied

N_FEATS = X_tr_final.shape[-1]

# El modelo de export tiene que replicar la ARQUITECTURA del modelo entrenado. Si el
# diagnostico eligio una config con otro numero de filtros (p.ej. D_chico -> 32), hay
# que propagarlo: con filters distintos no coincide ninguna shape del tronco y, ademas,
# con filters=32 sobre 32 features de entrada la proyeccion residual `tcn0_proj` ni
# siquiera se crea (prev.shape[-1] == filters). `dropout` y `l2` no importan aca: el
# modo export los fuerza a 0 y no afectan las shapes.
cfg_exp = {**CFG_BASE, **CFG_FINAL}
export_model = build_tcn_multitask((N_FRAMES, N_FEATS), export=True, batch_size=1,
                                   filters=cfg_exp['filters'])
print(f'Modelo de export con filters={cfg_exp["filters"]} '
      f'({export_model.count_params():,} parametros)')
copy_weights_by_name(best_model_31, export_model)

specs = [tf.TensorSpec(shape=(1, N_FRAMES, N_FEATS), dtype=tf.float32, name='features')]

tflite_float = keras_to_tflite(export_model, specs)
print(f'TFLite float: {len(tflite_float)/1024:.1f} KB')

# Cuantizacion int8 con dataset representativo real (~200 secuencias de Train)
rep_sample = X_tr_final[np.random.default_rng(SEED).choice(len(X_tr_final), size=200, replace=False)]
def representative_dataset():
    for i in range(len(rep_sample)):
        yield [rep_sample[i:i+1].astype('float32')]

tflite_int8 = keras_to_tflite(export_model, specs, representative_dataset)
print(f'TFLite int8: {len(tflite_int8)/1024:.1f} KB  (umbral del plan: < 500 KB)')

with open(f'{CKPT_DIR}/tcn_multitask_binaria_ordinal.tflite', 'wb') as f:
    f.write(tflite_int8)
with open(f'{CKPT_DIR}/tcn_multitask_binaria_ordinal_float.tflite', 'wb') as f:
    f.write(tflite_float)

### Verificación de la conversión — coherencia Keras vs. intérprete TFLite

Se compara la predicción binaria del modelo Keras contra la del intérprete TFLite sobre las mismas muestras de Test. El criterio del plan es que el delta de exactitud por cuantización quede por debajo de 1,5 pts; si es mayor, corresponde evaluar QAT en Fase 3.3.

Nota: con `from_saved_model` el orden de las salidas lo define la firma exportada y **no** siempre coincide con el orden de `keras.Model(inp, [binary, coral, attn])`. Por eso la celda identifica la salida binaria por su forma `(1, 1)` en vez de asumir el índice 0.

In [ ]:
interpreter = tf.lite.Interpreter(model_content=tflite_int8)
interpreter.allocate_tensors()
inp_details = interpreter.get_input_details()[0]
out_details = interpreter.get_output_details()

print('Entradas :', [(d['name'], d['shape'].tolist(), d['dtype'].__name__) for d in interpreter.get_input_details()])
print('Salidas  :', [(d['name'], d['shape'].tolist(), d['dtype'].__name__) for d in out_details])

# La cabeza binaria es la unica salida escalar por muestra -> shape (1, 1)
bin_idx = next(i for i, d in enumerate(out_details) if tuple(d['shape'][1:]) == (1,))
bin_det = out_details[bin_idx]
print(f'Salida binaria = indice {bin_idx} ("{bin_det["name"]}")')

n_check = min(300, len(X_te_s))
preds_tflite = np.zeros(n_check, dtype=np.float32)
for i in range(n_check):
    x_in = X_te_s[i:i+1].astype(np.float32)
    if inp_details['dtype'] == np.int8:
        s, zp = inp_details['quantization']
        x_in = np.clip(np.round(x_in / s + zp), -128, 127).astype(np.int8)
    interpreter.set_tensor(inp_details['index'], x_in)
    interpreter.invoke()
    out = interpreter.get_tensor(bin_det['index']).ravel()[0]
    if bin_det['dtype'] == np.int8:
        s_o, zp_o = bin_det['quantization']
        out = (float(out) - zp_o) * s_o
    preds_tflite[i] = out

acc_tflite = accuracy_score(y_te_final['y_bin'][:n_check], (preds_tflite > 0.5).astype(int))
acc_keras  = accuracy_score(y_te_final['y_bin'][:n_check], p_te_best[:n_check])
print(f'\nAccuracy TFLite int8 (primeras {n_check} muestras de Test): {acc_tflite:.4f}')
print(f'Accuracy Keras      (mismas {n_check} muestras):            {acc_keras:.4f}')
print(f'Delta por cuantizacion: {abs(acc_tflite - acc_keras):.4f} '
      f'({"OK, < 1,5 pts" if abs(acc_tflite - acc_keras) < 0.015 else "REVISAR: considerar QAT (Fase 3.3)"})')

with open(f'{CKPT_DIR}/fase3_1_tflite_reporte.json', 'w') as f:
    json.dump({'tflite_float_kb': round(len(tflite_float)/1024, 1),
               'tflite_int8_kb': round(len(tflite_int8)/1024, 1),
               'acc_keras': float(acc_keras), 'acc_tflite_int8': float(acc_tflite),
               'delta': float(abs(acc_tflite - acc_keras)), 'n_muestras': int(n_check)}, f, indent=2)

## 5. Criterios de salida de Fase 3.1

| Métrica | Umbral del plan | Resultado |
| :--- | :--- | :--- |
| Accuracy binaria Test oficial (media 5 semillas) | ≥ 60–63% | ver celda de resumen arriba |
| TFLite int8 | < 500 kB, funcionando | ver tamaño impreso arriba (típicamente ~100-150 kB para este tronco TCN) |
| Delta de exactitud por cuantización | Idealmente < 1,5 pts | ver celda anterior; si es mayor, considerar QAT (Fase 3.3) |

Si se cumple, corresponde avanzar a **Fase 3.2** (rama de apariencia con CNN ultra-liviana — la palanca principal según el diagnóstico del plan, §2). Si no se cumple con claridad: (a) probar el Transformer liviano alternativo (§3.2 del plan) antes de invertir en la rama de apariencia, (b) revisar si el peso de la pérdida ordinal (0,5) o auxiliar (0,1) está compitiendo demasiado con la binaria — barrer esos pesos, (c) revisar si `dropout=0.3` es demasiado agresivo para el tamaño de dataset tras la augmentación.